# YZTA 2026 Datathon - Final Solution

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from catboost import CatBoostRegressor, Pool
import lightgbm as lgb

warnings.filterwarnings("ignore")

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

RANDOM_STATE = 2026
K = 30

In [ ]:
# Dosya yolları.

def find_file(name):
    candidates = [Path(name), Path("tmp_data") / name]
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        candidates.extend(kaggle_root.rglob(name))
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"{name} bulunamadı")

train = pd.read_csv(find_file("train.csv"))
test = pd.read_csv(find_file("test_x.csv"))
ext = pd.read_csv(find_file("sleep_health_dataset.csv"))

target = "bilissel_performans_skoru"
y = train[target].to_numpy()
ext_y = ext["cognitive_performance_score"].to_numpy() / 10.0

print(train.shape, test.shape, ext.shape)

In [ ]:
occupation_map = {
    "Doctor": "Saglik Personeli",
    "Nurse": "Saglik Personeli",
    "Driver": "Lojistik Calisani",
    "Freelancer": "Serbest Calisan",
    "Homemaker": "Ev Hanimi",
    "Lawyer": "Lawyer",
    "Manager": "Yonetici",
    "Retired": "Emekli",
    "Sales": "Satis ve Pazarlama Calisani",
    "Software Engineer": "Muhendis",
    "Student": "Ogrenci",
    "Teacher": "Egitimci",
}
country_map = dict(zip(ext["country"].value_counts().index.tolist(), train["ulke"].value_counts().index.tolist()))
value_maps = {
    "occupation": occupation_map,
    "country": country_map,
    "gender": {"Male": "Erkek", "Female": "Kadin", "Other": "Erkek"},
    "chronotype": {"Morning": "Sabah insani", "Evening": "Gece insani", "Neutral": "Notr"},
    "mental_health_condition": {"Healthy": "Saglikli", "Anxiety": "Anksiyete", "Depression": "Depresyon", "Both": "Anksiyete ve depresyon"},
    "season": {"Spring": "Ilkbahar-Yaz", "Summer": "Ilkbahar-Yaz", "Autumn": "Sonbahar-Kis", "Winter": "Sonbahar-Kis"},
    "day_type": {"Weekday": "Hafta ici", "Weekend": "Hafta sonu"},
}
col_map = {
    "age": "yas",
    "gender": "cinsiyet",
    "occupation": "meslek",
    "bmi": "vucut_kitle_indeksi",
    "country": "ulke",
    "rem_percentage": "rem_yuzdesi",
    "deep_sleep_percentage": "derin_uyku_yuzdesi",
    "sleep_latency_mins": "uykuya_dalma_suresi_dk",
    "wake_episodes_per_night": "gecelik_uyanma_sayisi",
    "caffeine_mg_before_bed": "uyku_oncesi_kafein_mg",
    "screen_time_before_bed_mins": "uyku_oncesi_ekran_suresi_dk",
    "steps_that_day": "gunluk_adim_sayisi",
    "nap_duration_mins": "sekerleme_suresi_dk",
    "stress_score": "stres_skoru",
    "work_hours_that_day": "gunluk_calisma_saati",
    "chronotype": "kronotip",
    "mental_health_condition": "ruh_sagligi_durumu",
    "heart_rate_resting_bpm": "dinlenik_nabiz_bpm",
    "room_temperature_celsius": "oda_sicakligi_celsius",
    "weekend_sleep_diff_hrs": "hafta_sonu_uyku_farki_saat",
    "season": "mevsim",
    "day_type": "gun_tipi",
}
inv_col_map = {v: k for k, v in col_map.items()}

ext_visible = pd.DataFrame(index=ext.index)
for ext_col, local_col in col_map.items():
    s = ext[ext_col]
    if ext_col in value_maps:
        s = s.map(value_maps[ext_col])
    ext_visible[local_col] = s

local_all = pd.concat([train.drop(columns=[target]), test], ignore_index=True)
cat_cols = ["cinsiyet", "meslek", "ulke", "kronotip", "ruh_sagligi_durumu", "mevsim", "gun_tipi"]
num_cols = [c for c in ext_visible.columns if c not in cat_cols]

In [ ]:
# İlk 30 kaynak adayı: numeric standardize, categorical one-hot.
preprocess = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="Eksik")),
                      ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_cols),
])

X_ext = preprocess.fit_transform(ext_visible[num_cols + cat_cols])
X_local = preprocess.transform(local_all[num_cols + cat_cols])
knn = NearestNeighbors(n_neighbors=K, n_jobs=-1)
knn.fit(X_ext)
dist_all, ind_all = knn.kneighbors(X_local)

base_pred = ext_y[ind_all[:len(train), 0]]
top_targets = ext_y[ind_all[:len(train), :K]]
oracle_idx = np.argmin(np.abs(top_targets - y[:, None]), axis=1)
oracle_pred = top_targets[np.arange(len(train)), oracle_idx]

print("Nearest source RMSE:", rmse(y, base_pred))
print("Oracle among top 30 RMSE:", rmse(y, oracle_pred))

In [ ]:
# Proxy model: top-1 kaynak satırı + delta feature'ları ile doğrudan hedef tahmini.
hidden_cols = [
    "sleep_duration_hrs", "sleep_quality_score", "alcohol_units_before_bed", "exercise_day",
    "sleep_aid_used", "shift_work", "sleep_disorder_risk", "felt_rested",
]

def build_row_features(local_df, ind, dist):
    top = ext_y[ind]
    F = local_df.reset_index(drop=True).copy()
    F = F.drop(columns=["id"], errors="ignore")
    for k in range(10):
        F[f"nn_target_{k+1}"] = top[:, k]
        F[f"nn_dist_{k+1}"] = dist[:, k]
    for k in [3, 5, 10, 20, 30]:
        F[f"nn_target_mean_{k}"] = top[:, :k].mean(axis=1)
        F[f"nn_target_std_{k}"] = top[:, :k].std(axis=1)
        F[f"nn_target_min_{k}"] = top[:, :k].min(axis=1)
        F[f"nn_target_max_{k}"] = top[:, :k].max(axis=1)
    src_idx = ind[:, 0]
    for c in num_cols:
        e = inv_col_map[c]
        ev = ext[e].to_numpy()[src_idx]
        lv = local_df[c].to_numpy()
        F[f"ext_{c}"] = ev
        F[f"delta_{c}"] = lv - ev
        F[f"abs_delta_{c}"] = np.abs(lv - ev)
        F[f"miss_{c}"] = pd.isna(lv).astype(int)
    for c in cat_cols:
        ev = ext_visible[c].to_numpy()[src_idx]
        lv = local_df[c].astype(object).to_numpy()
        F[f"ext_{c}"] = ev
        F[f"match_{c}"] = (lv == ev).astype(int)
        F[f"miss_{c}"] = pd.isna(lv).astype(int)
    for c in hidden_cols:
        F[f"hidden_{c}"] = ext[c].to_numpy()[src_idx]
    return F

X_proxy = build_row_features(local_all.iloc[:len(train)], ind_all[:len(train)], dist_all[:len(train)])
X_proxy_test = build_row_features(local_all.iloc[len(train):], ind_all[len(train):], dist_all[len(train):])
cat_features_proxy = [i for i, c in enumerate(X_proxy.columns) if X_proxy[c].dtype == "object"]
for c in X_proxy.columns:
    if X_proxy[c].dtype == "object":
        X_proxy[c] = X_proxy[c].fillna("Eksik").astype(str)
        X_proxy_test[c] = X_proxy_test[c].fillna("Eksik").astype(str)

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof_proxy = np.zeros(len(train))
test_proxy = np.zeros(len(test))
proxy_params = dict(
    loss_function="RMSE", eval_metric="RMSE", iterations=2500, learning_rate=0.035,
    depth=6, l2_leaf_reg=5.0, random_seed=RANDOM_STATE, verbose=False,
    allow_writing_files=False, thread_count=-1,
)
for fold, (tr, va) in enumerate(kf.split(X_proxy), 1):
    model = CatBoostRegressor(**proxy_params)
    model.fit(
        Pool(X_proxy.iloc[tr], y[tr], cat_features=cat_features_proxy),
        eval_set=Pool(X_proxy.iloc[va], y[va], cat_features=cat_features_proxy),
        early_stopping_rounds=160,
        use_best_model=True,
        verbose=False,
    )
    oof_proxy[va] = model.predict(Pool(X_proxy.iloc[va], cat_features=cat_features_proxy))
    test_proxy += model.predict(Pool(X_proxy_test, cat_features=cat_features_proxy)) / kf.n_splits
    print(f"fold {fold}: {rmse(y[va], oof_proxy[va]):.5f}")
print("Proxy RMSE:", rmse(y, oof_proxy))

In [ ]:
# Pair feature matrisi: her satır x 30 aday.
loc_num = local_all[num_cols].to_numpy(dtype=np.float32)
ext_num = ext_visible[num_cols].to_numpy(dtype=np.float32)
loc_miss = np.isnan(loc_num).astype(np.float32)
med = np.nanmedian(ext_num, axis=0).astype(np.float32)
loc_num_imp = np.where(np.isnan(loc_num), med, loc_num).astype(np.float32)
ext_num_imp = np.where(np.isnan(ext_num), med, ext_num).astype(np.float32)

loc_cat, ext_cat, loc_cat_miss = [], [], []
for c in cat_cols:
    all_vals = pd.concat([local_all[c], ext_visible[c]], ignore_index=True).astype(object).fillna("Eksik")
    code_map = {v: i for i, v in enumerate(pd.unique(all_vals))}
    loc_cat.append(local_all[c].astype(object).fillna("Eksik").map(code_map).to_numpy(np.float32))
    ext_cat.append(ext_visible[c].astype(object).fillna("Eksik").map(code_map).to_numpy(np.float32))
    loc_cat_miss.append(local_all[c].isna().to_numpy(np.float32))
loc_cat = np.vstack(loc_cat).T
ext_cat = np.vstack(ext_cat).T
loc_cat_miss = np.vstack(loc_cat_miss).T

hidden = []
for c in hidden_cols:
    s = ext[c]
    if pd.api.types.is_numeric_dtype(s):
        hidden.append(s.to_numpy(np.float32))
    else:
        code_map = {v: i for i, v in enumerate(pd.unique(s.astype(object).fillna("Eksik")))}
        hidden.append(s.astype(object).fillna("Eksik").map(code_map).to_numpy(np.float32))
hidden = np.vstack(hidden).T.astype(np.float32)

rank_base = np.arange(K, dtype=np.float32) + 1
train_top = ext_y[ind_all[:len(train), :K]]
test_top = ext_y[ind_all[len(train):, :K]]

def row_stats(inds):
    top = ext_y[inds[:, :K]]
    return np.vstack([
        top[:, 0], top[:, :3].mean(1), top[:, :5].mean(1), top[:, :10].mean(1), top[:, :30].mean(1),
        top[:, :5].std(1), top[:, :10].std(1), top[:, :30].std(1), top[:, :30].min(1), top[:, :30].max(1),
    ]).T.astype(np.float32)

train_stats = row_stats(ind_all[:len(train)])
test_stats = row_stats(ind_all[len(train):])

def make_pair_features(rows, proxy, stats, is_train_part=True, reduced=False):
    rows = np.asarray(rows, dtype=np.int64)
    inds = ind_all[rows, :K]
    d = dist_all[rows, :K].astype(np.float32)
    cand = inds.reshape(-1)
    local_idx = rows if is_train_part else rows - len(train)
    p = np.repeat(proxy[local_idx].astype(np.float32), K)
    ct = ext_y[cand].astype(np.float32)
    rank = np.tile(rank_base, len(rows))
    dflat = d.reshape(-1)
    d1 = np.repeat(d[:, 0], K)
    ln = np.repeat(loc_num_imp[rows], K, axis=0)
    en = ext_num_imp[cand]
    lm = np.repeat(loc_miss[rows], K, axis=0)
    lc = np.repeat(loc_cat[rows], K, axis=0)
    ec = ext_cat[cand]
    hc = hidden[cand]
    if reduced:
        small_stats = np.repeat(np.vstack([
            ext_y[inds[:, :K]][:, 0], ext_y[inds[:, :K]][:, :5].mean(1), ext_y[inds[:, :K]][:, :30].std(1),
        ]).T.astype(np.float32), K, axis=0)
        blocks = [rank[:, None], dflat[:, None], (dflat - d1)[:, None], (dflat / (d1 + 1e-6))[:, None],
                  ct[:, None], p[:, None], np.abs(ct - p)[:, None], small_stats,
                  ln, en, ln - en, np.abs(ln - en), lm, lc, ec, (lc == ec).astype(np.float32), hc]
    else:
        rs = np.repeat(stats[local_idx], K, axis=0)
        dmean = np.repeat(d.mean(1), K)
        dstd = np.repeat(d.std(1), K)
        lcm = np.repeat(loc_cat_miss[rows], K, axis=0)
        blocks = [rank[:, None], dflat[:, None], (dflat - d1)[:, None], (dflat / (d1 + 1e-6))[:, None],
                  dmean[:, None], dstd[:, None], ct[:, None], p[:, None], (ct - p)[:, None], np.abs(ct - p)[:, None], rs,
                  ln, en, ln - en, np.abs(ln - en), lm, lc, ec, (lc == ec).astype(np.float32), lcm, hc]
    return np.hstack(blocks).astype(np.float32)

def oracle_class_labels(rows):
    top = ext_y[ind_all[rows, :K]]
    best = np.argmin(np.abs(top - y[rows, None]), axis=1)
    labels = np.zeros(len(rows) * K, dtype=np.int8)
    labels[np.arange(len(rows)) * K + best] = 1
    return labels

def bucket_rank_labels(rows):
    top = ext_y[ind_all[rows, :K]]
    err = np.abs(top - y[rows, None])
    return np.clip(np.floor((0.35 - err) / 0.025), 0, 14).astype(int).reshape(-1)

In [ ]:
# Aday sınıflandırıcı: 30 adaydan oracle'a en yakın hedefe sahip adayı öğrenir.
kf3 = KFold(n_splits=3, shuffle=True, random_state=731)
oof_clf = np.zeros(len(train))
clf_best_iters = []
for fold, (tr, va) in enumerate(kf3.split(np.arange(len(train))), 1):
    Xtr = make_pair_features(tr, oof_proxy, train_stats, True, reduced=False)
    ytr = oracle_class_labels(tr)
    Xva = make_pair_features(va, oof_proxy, train_stats, True, reduced=False)
    yva = oracle_class_labels(va)
    model = lgb.LGBMClassifier(
        objective="binary", n_estimators=2200, learning_rate=0.035, num_leaves=96,
        min_child_samples=60, subsample=0.9, colsample_bytree=0.9, reg_lambda=2.0,
        n_jobs=-1, random_state=100 + fold, scale_pos_weight=29, force_row_wise=True,
    )
    model.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric="auc",
              callbacks=[lgb.early_stopping(120, verbose=False), lgb.log_evaluation(0)])
    clf_best_iters.append(model.best_iteration_ or 7)
    prob = model.predict_proba(Xva, num_iteration=model.best_iteration_)[:, 1].reshape(len(va), K)
    pick = np.argmax(prob, axis=1)
    oof_clf[va] = train_top[va, pick]
    print(f"clf fold {fold}: {rmse(y[va], oof_clf[va]):.5f}")
print("Candidate classifier RMSE:", rmse(y, oof_clf), "best iters:", clf_best_iters)

# Final sınıflandırıcı: validasyona göre en iyi iterasyon 7 civarıdır.
X_all = make_pair_features(np.arange(len(train)), oof_proxy, train_stats, True, reduced=False)
y_all = oracle_class_labels(np.arange(len(train)))
X_test_pairs = make_pair_features(np.arange(len(train), len(train) + len(test)), test_proxy, test_stats, False, reduced=False)
clf_final = lgb.LGBMClassifier(
    objective="binary", n_estimators=int(np.median(clf_best_iters)), learning_rate=0.035, num_leaves=96,
    min_child_samples=60, subsample=0.9, colsample_bytree=0.9, reg_lambda=2.0,
    n_jobs=-1, random_state=20260515, scale_pos_weight=29, force_row_wise=True,
)
clf_final.fit(X_all, y_all, callbacks=[lgb.log_evaluation(0)])
prob_test = clf_final.predict_proba(X_test_pairs)[:, 1].reshape(len(test), K)
test_clf = test_top[np.arange(len(test)), np.argmax(prob_test, axis=1)]
print("classifier test mean:", test_clf.mean())

In [ ]:
# Bucket ranker: adayları hedefe yakınlık kovalarına göre doğrudan sıralar.
oof_ranker = np.zeros(len(train))
ranker_best_iters = []
for fold, (tr, va) in enumerate(kf3.split(np.arange(len(train))), 1):
    Xtr = make_pair_features(tr, oof_proxy, train_stats, True, reduced=True)
    ytr = bucket_rank_labels(tr)
    Xva = make_pair_features(va, oof_proxy, train_stats, True, reduced=True)
    yva = bucket_rank_labels(va)
    model = lgb.LGBMRanker(
        objective="lambdarank", n_estimators=500, learning_rate=0.05, num_leaves=63,
        min_child_samples=80, subsample=0.9, colsample_bytree=0.9, reg_lambda=2.0,
        n_jobs=-1, random_state=400 + fold, force_row_wise=True,
    )
    model.fit(Xtr, ytr, group=np.full(len(tr), K), eval_set=[(Xva, yva)], eval_group=[np.full(len(va), K)],
              eval_at=[1, 3], callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
    ranker_best_iters.append(model.best_iteration_ or 30)
    score = model.predict(Xva, num_iteration=model.best_iteration_).reshape(len(va), K)
    pick = np.argmax(score, axis=1)
    oof_ranker[va] = train_top[va, pick]
    print(f"ranker fold {fold}: {rmse(y[va], oof_ranker[va]):.5f}")
print("Ranker RMSE:", rmse(y, oof_ranker), "best iters:", ranker_best_iters)

X_rank_all = make_pair_features(np.arange(len(train)), oof_proxy, train_stats, True, reduced=True)
y_rank_all = bucket_rank_labels(np.arange(len(train)))
X_rank_test = make_pair_features(np.arange(len(train), len(train) + len(test)), test_proxy, test_stats, False, reduced=True)
ranker_final = lgb.LGBMRanker(
    objective="lambdarank", n_estimators=int(np.median(ranker_best_iters)), learning_rate=0.05, num_leaves=63,
    min_child_samples=80, subsample=0.9, colsample_bytree=0.9, reg_lambda=2.0,
    n_jobs=-1, random_state=20260516, force_row_wise=True,
)
ranker_final.fit(X_rank_all, y_rank_all, group=np.full(len(train), K), callbacks=[lgb.log_evaluation(0)])
score_test = ranker_final.predict(X_rank_test).reshape(len(test), K)
test_ranker = test_top[np.arange(len(test)), np.argmax(score_test, axis=1)]

In [ ]:
# Final blend: ranker + oracle-classifier + close-classifier.
blend_train = np.vstack([oof_ranker, oof_clf, oof_close, oof_proxy]).T
blend_test = np.vstack([test_ranker, test_clf, test_close, test_proxy]).T
res = minimize(
    lambda w: rmse(y, blend_train.dot(w)),
    np.ones(blend_train.shape[1]) / blend_train.shape[1],
    bounds=[(0, 1)] * blend_train.shape[1],
    constraints={"type": "eq", "fun": lambda w: np.sum(w) - 1},
    method="SLSQP",
)
print("Final blend RMSE:", res.fun)
print("Weights [ranker, classifier, close, proxy]:", res.x)

final_pred = blend_test.dot(res.x)
final_pred = np.clip(final_pred, 0, 10)
submission = pd.DataFrame({"id": test["id"], target: final_pred})
submission.to_csv("submission.csv", index=False)
submission.head()